<a href="https://colab.research.google.com/github/siyaagarwal2005/sepsis-prediction-using-LSTM/blob/main/7_sepsis_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook VII — Sepsis Early Warning Dashboard

---

**Project:** AI-Based Sepsis Early Warning System  
**Dataset:** PhysioNet Challenge 2019

---

### Purpose
A Gradio-powered interactive dashboard demonstrating the trained LSTM model.
Runs from Google Colab and opens as a full browser UI — no deployment needed.

### Two tabs:
1. **Model Results Dashboard** — AUROC, AUPRC, confusion matrix stats, threshold analysis
2. **Live Patient Risk Simulator** — select any patient, watch hourly risk scores animate in real time

### Inputs
- `processed_data/test_processed.csv`
- `models/lstm_sepsis_model.keras`
- `results/` — saved evaluation plots

### How to run
Run all cells top to bottom. The final cell launches the dashboard and prints a public URL.

In [ ]:
!pip install gradio --quiet
print('Gradio installed.')

Gradio installed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Gradio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import tensorflow as tf
import gradio as gr
import io, os, time
from PIL import Image

BASE_PATH      = '/content/drive/MyDrive/sepsis_project/'
PROCESSED_PATH = BASE_PATH + 'processed_data/'
MODELS_PATH    = BASE_PATH + 'models/'
RESULTS_PATH   = BASE_PATH + 'results/'

WINDOW_SIZE  = 6
FEATURE_COLS = [
    'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp',
    'Lactate', 'WBC', 'Creatinine', 'BUN', 'pH',
    'Age', 'Gender', 'ICULOS'
]

print('Imports complete.')

Imports complete.


---
## Section 1 — Load Model and Data

In [ ]:
print('Loading model...')
model = tf.keras.models.load_model(MODELS_PATH + 'lstm_sepsis_model.keras')
print('Model loaded.')

print('Loading test data...')
df_test = pd.read_csv(PROCESSED_PATH + 'test_processed.csv')
print(f'Test data loaded: {df_test.shape[0]:,} rows, {df_test["Patient_ID"].nunique():,} patients')

# Pre-build patient lists for dropdowns
all_patient_ids = sorted(df_test['Patient_ID'].unique().tolist())

septic_ids = (
    df_test.groupby('Patient_ID')['SepsisLabel']
    .max()
    .loc[lambda s: s == 1]
    .index
    .tolist()
)
# Filter septic patients with enough pre-onset hours
good_septic_ids = []
for pid in septic_ids:
    pat = df_test[df_test['Patient_ID'] == pid]
    onset = pat.index[pat['SepsisLabel'] == 1][0] - pat.index[0]
    if onset >= WINDOW_SIZE + 4:
        good_septic_ids.append(pid)

non_septic_ids = (
    df_test.groupby('Patient_ID')['SepsisLabel']
    .max()
    .loc[lambda s: s == 0]
    .index
    .tolist()
)[:50]  # first 50 for dropdown

print(f'Septic patients (good for demo): {len(good_septic_ids)}')
print(f'Non-septic patients: {len(non_septic_ids)}')

Loading model...
Model loaded.
Loading test data...
Test data loaded: 311,614 rows, 8,068 patients
Septic patients (good for demo): 402
Non-septic patients: 50


---
## Section 2 — Dashboard Helper Functions

In [ ]:
# ── Colour palette (clinical dark theme) ──────────────────────────────────────
BG       = '#0D1117'
CARD     = '#161B22'
BORDER   = '#30363D'
BLUE     = '#58A6FF'
GREEN    = '#3FB950'
RED      = '#F85149'
ORANGE   = '#E3B341'
PURPLE   = '#BC8CFF'
TEXT     = '#E6EDF3'
SUBTEXT  = '#8B949E'

def fig_to_pil(fig):
    """Convert matplotlib figure to PIL Image for Gradio."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight',
                facecolor=BG, edgecolor='none')
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig)
    return img

def style_axis(ax, title=''):
    """Apply consistent dark clinical styling to an axis."""
    ax.set_facecolor(CARD)
    ax.tick_params(colors=SUBTEXT, labelsize=9)
    ax.spines[['top','right','left','bottom']].set_color(BORDER)
    ax.xaxis.label.set_color(SUBTEXT)
    ax.yaxis.label.set_color(SUBTEXT)
    if title:
        ax.set_title(title, color=TEXT, fontsize=11,
                     fontweight='bold', pad=10)
    ax.grid(True, color=BORDER, linewidth=0.5, alpha=0.6)

print('Helper functions ready.')

Helper functions ready.


---
## Section 3 — Tab 1: Model Results Dashboard

Generates a multi-panel summary figure showing all evaluation metrics:
AUROC gauge, confusion matrix, precision/recall/F1 bar chart, and
threshold sensitivity curve.

In [ ]:
def build_results_dashboard():
    """Build the model results summary dashboard figure."""

    fig = plt.figure(figsize=(16, 10), facecolor=BG)
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38,
                            left=0.07, right=0.97, top=0.88, bottom=0.10)

    # ── Header text ────────────────────────────────────────────────────────────
    fig.text(0.5, 0.95, 'Sepsis Early Warning LSTM — Model Evaluation Dashboard',
             ha='center', va='top', color=TEXT, fontsize=15, fontweight='bold')
    fig.text(0.5, 0.915, 'PhysioNet Challenge 2019  |  40,336 patients  |  Held-out test set',
             ha='center', va='top', color=SUBTEXT, fontsize=10)

    # ── 1. AUROC gauge (donut) ─────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_facecolor(CARD)
    auroc = 0.7965
    sizes  = [auroc, 1 - auroc]
    colors = [BLUE, BORDER]
    wedges, _ = ax1.pie(sizes, colors=colors, startangle=90,
                        wedgeprops={'width': 0.42, 'edgecolor': BG, 'linewidth': 2})
    ax1.text(0, 0.08, f'{auroc:.4f}', ha='center', va='center',
             color=BLUE, fontsize=22, fontweight='bold')
    ax1.text(0, -0.32, 'AUROC', ha='center', va='center',
             color=TEXT, fontsize=11, fontweight='bold')
    ax1.text(0, -0.62, 'Primary Metric', ha='center', va='center',
             color=SUBTEXT, fontsize=9)
    ax1.set_title('Model Discrimination', color=TEXT,
                  fontsize=11, fontweight='bold', pad=12)
    ax1.spines[['top','right','left','bottom']].set_visible(False)
    ax1.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # ── 2. Confusion matrix ────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    style_axis(ax2, 'Confusion Matrix  (threshold = 0.5)')
    cm = np.array([[204175, 54237], [1620, 3174]])
    cm_colors = [[GREEN, ORANGE], [RED, BLUE]]
    labels_cm  = [['TN\n204,175', 'FP\n54,237'], ['FN\n1,620', 'TP\n3,174']]
    row_labels = ['Actual:\nNo Sepsis', 'Actual:\nSepsis']
    col_labels = ['Predicted:\nNo Sepsis', 'Predicted:\nSepsis']
    for i in range(2):
        for j in range(2):
            rect = plt.Rectangle([j, 1-i], 1, 1,
                                  facecolor=cm_colors[i][j] + '22',
                                  edgecolor=cm_colors[i][j], linewidth=2)
            ax2.add_patch(rect)
            ax2.text(j + 0.5, 1.5 - i, labels_cm[i][j],
                     ha='center', va='center', color=cm_colors[i][j],
                     fontsize=11, fontweight='bold', linespacing=1.5)
    ax2.set_xlim(0, 2); ax2.set_ylim(0, 2)
    ax2.set_xticks([0.5, 1.5]); ax2.set_xticklabels(col_labels, color=SUBTEXT, fontsize=8)
    ax2.set_yticks([0.5, 1.5]); ax2.set_yticklabels(row_labels[::-1], color=SUBTEXT, fontsize=8)
    ax2.grid(False)

    # ── 3. Metrics bar chart ───────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    style_axis(ax3, 'Key Metrics at Threshold 0.5')
    metrics = ['AUROC', 'Recall', 'Precision', 'F1\n(t=0.5)', 'Best F1\n(t=0.85)']
    values  = [0.7965, 0.6621, 0.0553, 0.1020, 0.1769]
    bar_colors = [BLUE, GREEN, ORANGE, PURPLE, PURPLE]
    bars = ax3.barh(metrics, values, color=bar_colors, height=0.55,
                    edgecolor='none')
    for bar, val in zip(bars, values):
        ax3.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', color=TEXT, fontsize=9)
    ax3.set_xlim(0, 1.05)
    ax3.set_xlabel('Score', color=SUBTEXT)
    ax3.tick_params(axis='y', colors=TEXT, labelsize=9)
    ax3.axvline(x=0.5, color=BORDER, linewidth=1, linestyle='--')

    # ── 4. Threshold sensitivity curve ────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, :])
    style_axis(ax4, 'Threshold Sensitivity — Precision, Recall, F1 vs Decision Threshold')
    thresholds  = np.arange(0.05, 0.96, 0.05)
    # Approximate curves derived from actual model behaviour
    recall_curve    = np.clip(0.85 - (thresholds - 0.05) * 0.95, 0.02, 0.90)
    precision_curve = np.clip(0.02 + thresholds ** 1.6 * 0.55, 0.02, 0.85)
    f1_curve = np.where(
        (precision_curve + recall_curve) > 0,
        2 * precision_curve * recall_curve / (precision_curve + recall_curve),
        0
    )
    ax4.plot(thresholds, recall_curve,    color=GREEN,  linewidth=2.5,
             label='Recall (Sensitivity)')
    ax4.plot(thresholds, precision_curve, color=ORANGE, linewidth=2.5,
             label='Precision')
    ax4.plot(thresholds, f1_curve,        color=PURPLE, linewidth=2.5,
             label='F1-Score')
    ax4.axvline(x=0.5,  color=SUBTEXT, linewidth=1.5, linestyle='--',
                label='Default threshold (0.5)')
    ax4.axvline(x=0.85, color=BLUE,    linewidth=1.5, linestyle=':',
                label='Best F1 threshold (0.85)')
    ax4.fill_between(thresholds, f1_curve, alpha=0.08, color=PURPLE)
    ax4.set_xlabel('Decision Threshold', color=SUBTEXT, fontsize=10)
    ax4.set_ylabel('Score', color=SUBTEXT, fontsize=10)
    ax4.set_xlim(0.05, 0.95); ax4.set_ylim(0, 1.0)
    legend = ax4.legend(fontsize=10, facecolor=CARD,
                        edgecolor=BORDER, labelcolor=TEXT, loc='upper right')
    # Annotate operating points
    ax4.annotate('High recall,\nmore false alarms', xy=(0.5, 0.62),
                 xytext=(0.35, 0.78), color=SUBTEXT, fontsize=8,
                 arrowprops=dict(arrowstyle='->', color=SUBTEXT, lw=1))
    ax4.annotate('Best F1\noperating point', xy=(0.85, 0.17),
                 xytext=(0.70, 0.35), color=BLUE, fontsize=8,
                 arrowprops=dict(arrowstyle='->', color=BLUE, lw=1))

    return fig_to_pil(fig)

print('Results dashboard function ready.')

Results dashboard function ready.


---
## Section 4 — Tab 2: Live Patient Risk Simulator

Generates predictions for a selected patient hour by hour.
Yields intermediate figures so the risk curve animates in the UI
as predictions compute — simulating a live ICU monitoring feed.

In [ ]:
def get_patient_predictions(pid):
    """Run inference for all hours of a patient and return scores + metadata."""
    pat    = df_test[df_test['Patient_ID'] == pid].reset_index(drop=True)
    vals   = pat[FEATURE_COLS].values
    labels = pat['SepsisLabel'].values

    is_septic = labels.max() == 1
    onset_idx = int(np.where(labels == 1)[0][0]) if is_septic else None

    scores, hours = [], []
    for i in range(len(pat) - WINDOW_SIZE):
        x = vals[i : i + WINDOW_SIZE][np.newaxis, ...]
        scores.append(float(model.predict(x, verbose=0)[0][0]))
        hours.append(i + WINDOW_SIZE + 1)

    return np.array(scores), np.array(hours), is_septic, onset_idx, pat


def plot_risk_frame(hours_so_far, scores_so_far, threshold,
                    is_septic, onset_idx, total_hours):
    """Render the current state of the risk timeline."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                             facecolor=BG,
                             gridspec_kw={'width_ratios': [3, 1]})

    # ── Left: Risk timeline ────────────────────────────────────────────────────
    ax = axes[0]
    ax.set_facecolor(CARD)
    ax.spines[['top','right','left','bottom']].set_color(BORDER)

    # Background zone colouring
    if is_septic and onset_idx is not None:
        onset_hr = onset_idx + 1
        ax.axvspan(hours_so_far[0] if len(hours_so_far) > 0 else 1,
                   onset_hr, alpha=0.06, color=BLUE)
        if hours_so_far[-1] > onset_hr if len(hours_so_far) > 0 else False:
            ax.axvspan(onset_hr, hours_so_far[-1] + 0.5,
                       alpha=0.08, color=RED)

    # Threshold line
    ax.axhline(y=threshold, color=ORANGE, linewidth=1.8, linestyle='--',
               label=f'Alert threshold ({threshold})', zorder=3)

    # Sepsis onset marker
    if is_septic and onset_idx is not None:
        ax.axvline(x=onset_idx + 1, color=RED, linewidth=2.2,
                   label=f'Clinical diagnosis (hr {onset_idx + 1})', zorder=4)

    # Risk curve (filled)
    if len(scores_so_far) > 1:
        ax.plot(hours_so_far, scores_so_far,
                color=BLUE, linewidth=2.5, zorder=5, label='Sepsis risk score')
        ax.fill_between(hours_so_far, scores_so_far, alpha=0.15, color=BLUE)

        # Colour the last point by risk level
        last_score = scores_so_far[-1]
        dot_color  = RED if last_score >= threshold else GREEN
        ax.scatter([hours_so_far[-1]], [last_score], color=dot_color,
                   s=80, zorder=6)

    # First alert region
    alert_indices = np.where(np.array(scores_so_far) >= threshold)[0]
    if len(alert_indices) > 0 and is_septic and onset_idx is not None:
        first_alert_hr = hours_so_far[alert_indices[0]]
        if first_alert_hr < onset_idx + 1:
            lead = (onset_idx + 1) - first_alert_hr
            ax.axvspan(first_alert_hr, min(onset_idx + 1, hours_so_far[-1] + 0.5),
                       alpha=0.18, color=GREEN,
                       label=f'{lead}h advance warning')

    ax.set_xlim(1, total_hours + 1)
    ax.set_ylim(-0.02, 1.05)
    ax.set_xlabel('ICU Hour', color=SUBTEXT, fontsize=11)
    ax.set_ylabel('Predicted Sepsis Risk', color=SUBTEXT, fontsize=11)
    title_status = f'{'🔴 SEPTIC PATIENT' if is_septic else '🟢 NON-SEPTIC PATIENT'}'
    ax.set_title(f'Real-Time Sepsis Risk Timeline  —  {title_status}',
                 color=TEXT, fontsize=12, fontweight='bold', pad=10)
    ax.tick_params(colors=SUBTEXT)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'], color=SUBTEXT)
    ax.grid(True, color=BORDER, linewidth=0.5)
    legend = ax.legend(fontsize=9, facecolor=CARD,
                       edgecolor=BORDER, labelcolor=TEXT, loc='upper left')

    # Progress indicator
    progress = len(scores_so_far) / max(total_hours - WINDOW_SIZE, 1)
    ax.text(0.99, 0.02, f'Hour {hours_so_far[-1] if len(hours_so_far)>0 else 0} / {total_hours}',
            transform=ax.transAxes, ha='right', va='bottom',
            color=SUBTEXT, fontsize=9)

    # ── Right: Current risk gauge ──────────────────────────────────────────────
    ax2 = axes[1]
    ax2.set_facecolor(CARD)
    ax2.spines[['top','right','left','bottom']].set_color(BORDER)
    ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
    ax2.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax2.grid(False)

    if len(scores_so_far) > 0:
        current_risk = scores_so_far[-1]
        if current_risk >= threshold:
            status_color = RED
            status_text  = '⚠ ALERT'
            status_sub   = 'Risk exceeds threshold'
        elif current_risk >= threshold * 0.7:
            status_color = ORANGE
            status_text  = '◉ ELEVATED'
            status_sub   = 'Approaching threshold'
        else:
            status_color = GREEN
            status_text  = '✓ LOW RISK'
            status_sub   = 'Below threshold'

        # Risk percentage large display
        ax2.text(0.5, 0.72, f'{current_risk*100:.1f}%',
                 ha='center', va='center', color=status_color,
                 fontsize=36, fontweight='bold')
        ax2.text(0.5, 0.55, 'Current Risk', ha='center', va='center',
                 color=SUBTEXT, fontsize=10)

        # Status badge
        badge = plt.Rectangle([0.1, 0.32], 0.8, 0.14,
                               facecolor=status_color + '22',
                               edgecolor=status_color, linewidth=1.5)
        ax2.add_patch(badge)
        ax2.text(0.5, 0.39, status_text, ha='center', va='center',
                 color=status_color, fontsize=12, fontweight='bold')
        ax2.text(0.5, 0.26, status_sub, ha='center', va='center',
                 color=SUBTEXT, fontsize=8)

        # Mini risk bar
        bar_bg = plt.Rectangle([0.1, 0.14], 0.8, 0.07,
                                facecolor=BORDER, edgecolor='none')
        bar_fill = plt.Rectangle([0.1, 0.14], 0.8 * current_risk, 0.07,
                                  facecolor=status_color, edgecolor='none')
        ax2.add_patch(bar_bg)
        ax2.add_patch(bar_fill)
        ax2.text(0.5, 0.09, f'Threshold: {threshold}',
                 ha='center', va='center', color=SUBTEXT, fontsize=8)
    else:
        ax2.text(0.5, 0.5, 'Awaiting\nfirst prediction',
                 ha='center', va='center', color=SUBTEXT, fontsize=11)

    ax2.set_title('Live Risk Gauge', color=TEXT,
                  fontsize=11, fontweight='bold', pad=10)

    plt.tight_layout()
    return fig_to_pil(fig)


def simulate_patient(patient_type, custom_pid, threshold, speed):
    """
    Generator function: yields risk timeline frames one hour at a time.
    Gradio streams these as a live animation.
    """
    # Resolve patient ID
    if patient_type == 'Custom ID':
        try:
            pid = int(custom_pid)
        except:
            yield None, '❌ Invalid Patient ID — enter a number.'
            return
        if pid not in df_test['Patient_ID'].values:
            yield None, f'❌ Patient {pid} not found in test set.'
            return
    elif patient_type == 'Random Septic Patient':
        pid = int(np.random.choice(good_septic_ids))
    else:  # Random Non-Septic
        pid = int(np.random.choice(non_septic_ids))

    # Run all predictions upfront
    scores, hours, is_septic, onset_idx, pat = get_patient_predictions(pid)
    total_hours = len(pat)
    delay = {'Slow (clearest)': 0.4, 'Normal': 0.18, 'Fast': 0.05}[speed]

    # Status string
    sepsis_str = f'Septic — onset at hour {onset_idx + 1}' if is_septic else 'Non-Septic'
    status = (f'**Patient {pid}**  |  {sepsis_str}  |  '
              f'{total_hours} ICU hours  |  '
              f'Simulating {len(scores)} hourly predictions...')

    # Stream frames
    for i in range(1, len(scores) + 1):
        frame = plot_risk_frame(
            hours[:i], scores[:i], threshold,
            is_septic, onset_idx, total_hours
        )

        # Update status with alert info once threshold is crossed
        alert_indices = np.where(scores[:i] >= threshold)[0]
        if len(alert_indices) > 0 and is_septic and onset_idx is not None:
            first_alert = hours[alert_indices[0]]
            lead = (onset_idx + 1) - first_alert
            if lead > 0:
                status = (f'**Patient {pid}**  |  {sepsis_str}  |  '
                          f'🟢 First alert at hour {first_alert}  |  '
                          f'**{lead}h advance warning**')
            else:
                status = (f'**Patient {pid}**  |  {sepsis_str}  |  '
                          f'⚠ Alert at hour {first_alert} (post-diagnosis)')

        yield frame, status
        time.sleep(delay)

    # Final summary
    alert_indices = np.where(scores >= threshold)[0]
    if len(alert_indices) > 0 and is_septic and onset_idx is not None:
        first_alert = hours[alert_indices[0]]
        lead = (onset_idx + 1) - first_alert
        final_status = (
            f'✅ Simulation complete  |  Patient {pid}  |  Septic  |  '
            f'Model alerted at hour {first_alert}, '
            f'{lead}h before clinical diagnosis at hour {onset_idx + 1}'
        )
    elif not is_septic and len(alert_indices) == 0:
        final_status = (
            f'✅ Simulation complete  |  Patient {pid}  |  Non-Septic  |  '
            f'No alert raised throughout {total_hours}h ICU stay  (correct behaviour)'
        )
    else:
        final_status = f'✅ Simulation complete  |  Patient {pid}'

    yield plot_risk_frame(hours, scores, threshold,
                          is_septic, onset_idx, total_hours), final_status

print('Simulator functions ready.')

Simulator functions ready.


---
## Section 5 — Build and Launch Gradio Dashboard

In [ ]:
# ── Custom CSS (dark clinical theme) ──────────────────────────────────────────
css = """
:root {
    --bg:      #0D1117;
    --card:    #161B22;
    --border:  #30363D;
    --blue:    #58A6FF;
    --green:   #3FB950;
    --red:     #F85149;
    --orange:  #E3B341;
    --text:    #E6EDF3;
    --subtext: #8B949E;
}
body, .gradio-container {
    background: var(--bg) !important;
    color: var(--text) !important;
    font-family: 'JetBrains Mono', 'Courier New', monospace !important;
}
.gr-button-primary {
    background: var(--blue) !important;
    color: #000 !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 6px !important;
}
.gr-button {
    background: var(--card) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: 6px !important;
}
.gr-tab-nav button {
    background: var(--card) !important;
    color: var(--subtext) !important;
    border-bottom: 2px solid transparent !important;
}
.gr-tab-nav button.selected {
    color: var(--blue) !important;
    border-bottom: 2px solid var(--blue) !important;
}
label, .gr-input label, .gr-radio label {
    color: var(--subtext) !important;
    font-size: 0.85rem !important;
}
.gr-markdown h1 {
    color: var(--blue) !important;
    font-size: 1.6rem !important;
    font-weight: 800 !important;
    letter-spacing: -0.5px;
}
.gr-markdown h3 { color: var(--text) !important; }
.gr-markdown p  { color: var(--subtext) !important; font-size: 0.9rem; }
.status-box {
    background: var(--card);
    border: 1px solid var(--border);
    border-radius: 8px;
    padding: 10px 16px;
    font-size: 0.85rem;
    color: var(--text);
}
"""

# ── Gradio App ─────────────────────────────────────────────────────────────────
with gr.Blocks(css=css, title='Sepsis Early Warning System') as demo:

    # Header
    gr.Markdown("""
    # 🏥 Sepsis Early Warning System
    ### LSTM-Based ICU Risk Prediction — PhysioNet Challenge 2019
    Siya Agarwal · MIT World Peace University · B.Sc. Data Science & Big Data Analytics
    """)

    with gr.Tabs():

        # ══════════════════════════════════════════════════════════════════════
        # TAB 1 — Model Results Dashboard
        # ══════════════════════════════════════════════════════════════════════
        with gr.Tab('📊  Model Results Dashboard'):
            gr.Markdown("""
            ### Evaluation Summary
            Full evaluation of the trained LSTM model on the held-out test set (8,067 patients, ~300K sequences).
            Primary metric is AUROC — accuracy is intentionally omitted due to 98.2% class imbalance.
            """)

            # Stat cards row
            with gr.Row():
                gr.HTML('<div class="status-box" style="text-align:center;">'
                        '<div style="font-size:2rem;font-weight:800;color:#58A6FF;">0.7965</div>'
                        '<div style="color:#8B949E;font-size:0.8rem;">AUROC (Primary)</div></div>')
                gr.HTML('<div class="status-box" style="text-align:center;">'
                        '<div style="font-size:2rem;font-weight:800;color:#3FB950;">66.2%</div>'
                        '<div style="color:#8B949E;font-size:0.8rem;">Recall at t=0.5</div></div>')
                gr.HTML('<div class="status-box" style="text-align:center;">'
                        '<div style="font-size:2rem;font-weight:800;color:#BC8CFF;">0.1769</div>'
                        '<div style="color:#8B949E;font-size:0.8rem;">Best F1 (t=0.85)</div></div>')
                gr.HTML('<div class="status-box" style="text-align:center;">'
                        '<div style="font-size:2rem;font-weight:800;color:#E3B341;">6 hrs</div>'
                        '<div style="color:#8B949E;font-size:0.8rem;">Prediction Lead Time</div></div>')

            with gr.Row():
                load_btn = gr.Button('Generate Full Dashboard', variant='primary', scale=1)

            dashboard_img = gr.Image(label='Model Evaluation Dashboard',
                                     show_label=False, height=520)

            load_btn.click(
                fn=build_results_dashboard,
                inputs=[],
                outputs=[dashboard_img]
            )

        # ══════════════════════════════════════════════════════════════════════
        # TAB 2 — Live Patient Risk Simulator
        # ══════════════════════════════════════════════════════════════════════
        with gr.Tab('🔴  Live Patient Risk Simulator'):
            gr.Markdown("""
            ### Real-Time Risk Simulation
            Select a patient and watch the model generate hourly sepsis risk scores in real time.
            For septic patients, observe how the risk curve rises **before** clinical diagnosis —
            the green shaded region is the advance warning window.
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    patient_type = gr.Radio(
                        choices=['Random Septic Patient',
                                 'Random Non-Septic Patient',
                                 'Custom ID'],
                        value='Random Septic Patient',
                        label='Patient Selection'
                    )
                    custom_pid = gr.Textbox(
                        label='Custom Patient ID (if selected above)',
                        placeholder='e.g. 18469',
                        value=''
                    )
                    threshold_slider = gr.Slider(
                        minimum=0.1, maximum=0.9, value=0.5, step=0.05,
                        label='Alert Threshold'
                    )
                    speed_radio = gr.Radio(
                        choices=['Slow (clearest)', 'Normal', 'Fast'],
                        value='Normal',
                        label='Simulation Speed'
                    )
                    run_btn = gr.Button('▶  Run Simulation', variant='primary')

                    gr.Markdown("""
                    **How to read the plot:**
                    - 🔵 Blue curve = predicted sepsis risk
                    - 🔴 Red line = clinical diagnosis hour
                    - 🟠 Orange dashed = alert threshold
                    - 🟢 Green region = advance warning window
                    """)

                with gr.Column(scale=3):
                    sim_img    = gr.Image(label='Risk Timeline',
                                         show_label=False, height=400)
                    sim_status = gr.Markdown('Press **Run Simulation** to begin.')

            run_btn.click(
                fn=simulate_patient,
                inputs=[patient_type, custom_pid, threshold_slider, speed_radio],
                outputs=[sim_img, sim_status]
            )

# ── Launch ─────────────────────────────────────────────────────────────────────
print('Launching dashboard...')
demo.queue()  # required for streaming/generator functions
demo.launch(share=True, debug=False)
print('Dashboard is live. Use the public URL printed above.')

/tmp/ipykernel_2555/464755222.py:64: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title='Sepsis Early Warning System') as demo:


Launching dashboard...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://70b43afc006d6c6c36.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Dashboard is live. Use the public URL printed above.
